# SHAP Object Creation - Documented Example

This notebook provides a detailed, step-by-step walkthrough of how a single SHAP explanation object is created from the raw Bipolar HiPIMS experimental data. It serves as documentation for the data pipeline that produces the pickled SHAP objects used throughout the publication.

## Overview

The workflow consists of the following steps:
1. Load a BayBE Campaign from its serialized JSON file
2. Import calibration data (oscilloscope window, deposition rate calibration)
3. Extract peak current density (J_pk) from oscilloscope logfiles
4. Construct a clean feature DataFrame
5. Build a new BayBE Campaign with the desired parameter set
6. Generate SHAP explanations using BayBE's `SHAPInsight`
7. Pickle the resulting explanation object

## Notes on Data Preprocessing and Outlier Handling

**No preprocessing is applied to the data.** The raw measurement values are used directly without any normalization, standardization, or feature scaling prior to model training and SHAP explanation generation.

**No outlier detection or removal is performed.** All measured data points are included as-is. The BayBE surrogate model (Gaussian Process Regression) and the SHAP explainer operate on the complete, unfiltered dataset. This is a deliberate choice: the active learning loop already guides the sampling, and removing data points would bias the explanation of the surrogate model's behavior.

The only data transformations applied are:
- Unit conversion of deposition rate from kAngstrom/s to Angstrom/s (multiplication by 1000)
- Material-specific calibration factor applied to the deposition rate (density correction for Al vs Ti)
- Peak current density correction for the 3-inch target area (division by 45.58 cm²)

## Step 0: Imports

In [ ]:
import json
import pandas as pd
import numpy as np
import baybe
from baybe.insights import SHAPInsight
import os
import pickle
from pathlib import Path

import InitializeCampaign as ic
import shapexplainers as se

## Step 1: Define Paths and Select a Dataset

Each dataset corresponds to a single experimental campaign (material + power + pulse width regime). The folder contains:
- `Campaign.json` — serialized BayBE Campaign object with the full search space and measurements
- `calibration.txt` — oscilloscope sampling window (`us-window`) and material-specific deposition rate calibration factor (`dep-rate calibration`)
- `Logfile - Oscilloscope/` — individual oscilloscope waveform files (JSON) for each measurement, used for peak current extraction

We demonstrate with the **Al - 120 W - short PW** dataset.

In [ ]:
# Base path to the datasets
base_dir = Path.cwd()
HEAD_PATH = base_dir / "Bipolar Datasets - Al and Ti - low and high PW" / "Datasets Used for Publication"

# Select the dataset to process
DATASET_NAME = "Al - 120 W  - short PW"
FOLDER_PATH = HEAD_PATH / DATASET_NAME

# Define sub-paths
CAMPAIGN_PATH = FOLDER_PATH / 'Campaign.json'
OSCCI_PATH = FOLDER_PATH / 'Logfile - Oscilloscope'

print(f"Dataset folder: {FOLDER_PATH}")
print(f"Campaign file exists: {CAMPAIGN_PATH.exists()}")
print(f"Oscilloscope folder exists: {OSCCI_PATH.exists()}")

## Step 2: Load the BayBE Campaign

The `Campaign.json` file is a serialized BayBE Campaign that contains:
- The continuous search space definition (parameter names, lower/upper bounds)
- All experimental measurements collected during the active learning loop

The campaign's search space parameters are the process variables controlled during the experiment (e.g., pulse width, pulse repetition rate, positive pulse settings). Note that `Ipk` (peak current density) is **not** a controlled parameter — it is a measured response that depends on the other settings.

In [ ]:
# Load and deserialize the BayBE Campaign
json_str = open(CAMPAIGN_PATH).read()
jay = json.loads(json_str)
campaign = baybe.Campaign.from_json(jay)

# Inspect the parameter space
params = [p.name for p in campaign.parameters]
print(f"Campaign parameters: {params}")
print(f"Number of measurements: {len(campaign.measurements)}")
print(f"\nMeasurement columns: {list(campaign.measurements.columns)}")
campaign.measurements.head()

## Step 3: Load Calibration Data

Each dataset folder contains a `calibration.txt` with two values:
- **`us-window`**: The oscilloscope sampling window in microseconds. This is needed to convert sample indices to physical time units.
- **`dep-rate calibration`**: A material-system-specific calibration factor that accounts for the density difference between Al and Ti when converting the QCM signal to a deposition rate in Angstrom/s.

In [ ]:
# Load calibration parameters
df_cal = pd.read_csv(FOLDER_PATH / 'calibration.txt')
us_window = int(df_cal['us-window'].iloc[0])          # oscilloscope window in microseconds
dep_rate_calibration = float(df_cal[' dep-rate calibration'].iloc[0])  # density-based calibration

print(f"Oscilloscope window: {us_window} us")
print(f"Deposition rate calibration factor: {dep_rate_calibration}")

## Step 4: Extract Peak Current Density (J_pk) from Oscilloscope Data

The peak current density is extracted from each oscilloscope waveform file. Each JSON file contains a single voltage/current waveform captured during one HiPIMS pulse.

**Extraction procedure:**
1. Convert sample indices to time using the `us-window` calibration
2. Define a trigger region: the oscilloscope trigger fires at approximately sample 900. We exclude a window after the trigger to avoid initial oscillations.
3. The peak current is found within the window from `trigger + exclusion zone` to `trigger + exclusion zone + pulse width`
4. The raw current is divided by the 3-inch target area (45.58 cm²) to obtain peak current density in A/cm²

**No filtering or smoothing** is applied to the waveforms.

In [ ]:
# Extract Ipk from each oscilloscope logfile
Ipk_list = []
df_pw = campaign.measurements  # need PW for windowing

for idx, file in enumerate(sorted(os.listdir(OSCCI_PATH))):
    # Read the oscilloscope waveform
    df_osc = pd.read_json(OSCCI_PATH / file)
    
    # Convert sample count to time axis
    corr_time = us_window / len(df_osc)  # us per sample point
    df_osc['Time'] = np.linspace(0, int(len(df_osc) * corr_time), len(df_osc))
    
    # Define trigger-based windowing for peak current determination
    trigger = 900 * corr_time           # estimated trigger position in us
    trigger_exclude = 250 * corr_time   # exclusion zone after trigger (avoids ringing)
    cutoff_trigger = trigger + trigger_exclude
    cutoff_pulse = cutoff_trigger + df_pw['PW (us)'][idx]  # end of pulse window
    
    # Find peak current within the valid window
    df_mask = df_osc.loc[(df_osc['Time'] > cutoff_trigger) & (df_osc['Time'] < cutoff_pulse)]
    
    # Convert to peak current density [A/cm2] using 3-inch target area
    Ipk_list.append(np.max(df_mask[2]) / 45.58)

print(f"Extracted {len(Ipk_list)} Ipk values")
print(f"Ipk range: {min(Ipk_list):.2f} - {max(Ipk_list):.2f} A/cm²")

## Step 5: Construct the Feature DataFrame

We combine the campaign measurements with the extracted Ipk values and apply the deposition rate calibration. The final feature set used for SHAP analysis includes:

| Feature | Description | Unit |
|---------|-------------|------|
| `Ipk (A)` | Peak current density (measured, not controlled) | A/cm² |
| `PW (us)` | Negative pulse width | μs |
| `PRR (Hz)` | Pulse repetition rate (frequency) | Hz |
| `pos. Delay (us)` | Delay before positive (counter) pulse | μs |
| `pos. PW (us)` | Positive pulse width | μs |
| `pos. Setpoint (V)` | Positive pulse setpoint voltage | V |

The target variable `y1` is the deposition rate in Angstrom/s.

For datasets that use duty cycle instead of PRR as a search space parameter, PRR is computed as: `PRR = Duty_Cycle / (PW * 1e-6)`.

In [ ]:
# Build the DataFrame
df_campaign = campaign.measurements.copy()

# Add the measured peak current density
df_campaign['Ipk (A)'] = Ipk_list

# Calibrate the deposition rate: convert kA/s -> A/s and apply material calibration
df_campaign['y1'] = df_campaign['y1'] * 1000 * dep_rate_calibration

# Handle duty cycle datasets: compute PRR if not present
if 'PRR (Hz)' not in df_campaign.columns:
    df_campaign['PRR (Hz)'] = df_campaign['Duty Cycle (ratio)'] / (df_campaign['PW (us)'] * 1E-6)

# Define the parameter set for SHAP analysis
params = ['Ipk (A)', 'PW (us)', 'PRR (Hz)', 'pos. Delay (us)', 'pos. PW (us)', 'pos. Setpoint (V)']

# Build a clean DataFrame with only the features of interest + target
df_clean = df_campaign[params + ['y1']].copy()

print(f"Clean dataset shape: {df_clean.shape}")
print(f"\nFeature statistics:")
df_clean.describe()

## Step 6: Create a New BayBE Campaign and Add Measurements

A new BayBE Campaign is initialized with the desired parameter bounds (derived from the data) and the clean feature set. This campaign serves as the input for the SHAP explainer.

The campaign uses a `SingleTargetObjective` with `mode="MAX"` for the deposition rate target.

In [ ]:
# Compute parameter bounds from the data
lower_bounds = [df_clean[p].min() for p in params]
upper_bounds = [df_clean[p].max() for p in params]

print("Parameter bounds:")
for p, lb, ub in zip(params, lower_bounds, upper_bounds):
    print(f"  {p}: [{lb:.2f}, {ub:.2f}]")

# Initialize a new campaign with these bounds
campaign_new = ic.init_campaign(lower_bounds, upper_bounds, params)

# Add the clean measurements
campaign_new.add_measurements(df_clean)
print(f"\nCampaign initialized with {len(campaign_new.measurements)} measurements")

## Step 7: Generate SHAP Explanations

BayBE's `SHAPInsight` wraps the SHAP library to explain the surrogate model (Gaussian Process Regression) used internally by the campaign. The surrogate model is trained on the computational representation of the data (`use_comp_rep=True`).

We use the **ExactExplainer**, which computes exact SHAP values by enumerating all possible feature coalitions on the GPR model predictions.

The resulting `shap.Explanation` object contains:
- `.values` — SHAP values for each sample x feature
- `.base_values` — the expected model output (mean prediction)
- `.data` — the raw feature values
- `.feature_names` — the names of the features

In [ ]:
# Create the SHAP insight using the Exact explainer
insight = se.insight_exact(campaign_new)

# Generate the explanation
explanation = insight.explain()

print(f"Explanation shape: {explanation.values.shape}")
print(f"Feature names: {explanation.feature_names}")
print(f"Base value (mean prediction): {explanation.base_values[0]:.2f} Angstrom/s")

## Step 8: Pickle the Explanation Object

The explanation is saved as a pickle file for later use in visualization and analysis notebooks.

In [ ]:
# Save the pickled explanation
output_path = base_dir / 'Pickles' / 'Ipk-PW-PRR-posPulse' / f'{DATASET_NAME}.pkl'

with open(output_path, "wb") as f:
    pickle.dump(explanation, f)

print(f"Saved explanation to: {output_path}")

## Summary

This notebook demonstrated the full pipeline for creating a single SHAP explanation object:

1. Raw oscilloscope waveforms -> peak current density extraction (no filtering)
2. Campaign measurements + Ipk -> clean feature DataFrame (no preprocessing, no outlier removal)
3. BayBE Campaign with GPR surrogate -> SHAP ExactExplainer -> pickled explanation

The same procedure is repeated for all six datasets (Al and Ti at different power/PW regimes) to produce the full set of pickled SHAP objects in `Pickles/Ipk-PW-PRR-posPulse/`.

Combined dataset explanations (e.g., `Al_combined_dataset_withPRR.pkl`) are generated by concatenating the clean DataFrames from multiple campaigns before creating the SHAP insight.